In [1]:
!pip install pandas sqlalchemy pymysql


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\santh\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

file_path = "hotel_bookings.csv"

df = pd.read_csv(file_path)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 119390
Columns: 32


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [3]:
# Make a copy
hotel = df.copy()

# --------------------------------------------------
# 1. Remove columns that are mostly missing / not
#    necessary for our analysis
# --------------------------------------------------

hotel.drop(columns=["agent", "company"], inplace=True)


# --------------------------------------------------
# 2. Handle missing values
# --------------------------------------------------

# Only 4 missing children values
hotel["children"] = hotel["children"].fillna(0)

# 488 missing country values
hotel["country"] = hotel["country"].fillna("Unknown")


# --------------------------------------------------
# 3. Remove duplicate records
# --------------------------------------------------

hotel.drop_duplicates(inplace=True)


# --------------------------------------------------
# 4. Create proper arrival date
# --------------------------------------------------

hotel["arrival_date"] = pd.to_datetime(
    hotel["arrival_date_year"].astype(str) + "-" +
    hotel["arrival_date_month"] + "-" +
    hotel["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d"
)


# --------------------------------------------------
# 5. Create useful derived columns
# --------------------------------------------------

# Total nights stayed
hotel["total_nights"] = (
    hotel["stays_in_weekend_nights"] +
    hotel["stays_in_week_nights"]
)

# Total guests
hotel["total_guests"] = (
    hotel["adults"] +
    hotel["children"] +
    hotel["babies"]
)

# Total special requests + parking
hotel["total_extra_requests"] = (
    hotel["total_of_special_requests"] +
    hotel["required_car_parking_spaces"]
)


# --------------------------------------------------
# 6. Standardize column names
# --------------------------------------------------

hotel.columns = (
    hotel.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)


# --------------------------------------------------
# 7. Check the cleaned data
# --------------------------------------------------

print("Original rows:", len(df))
print("Cleaned rows:", len(hotel))
print("Columns:", len(hotel.columns))

print("\nMissing values:")
print(hotel.isnull().sum()[hotel.isnull().sum() > 0])

hotel.head()

Original rows: 119390
Cleaned rows: 87370
Columns: 34

Missing values:
Series([], dtype: int64)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,total_nights,total_guests,total_extra_requests
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01,0,2.0,0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01,0,2.0,0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-01,1,1.0,0
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-01,1,1.0,0
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,Transient,98.0,0,1,Check-Out,2015-07-03,2015-07-01,2,2.0,1


In [6]:

from sqlalchemy import create_engine, text

# -----------------------------
# MySQL connection details
# -----------------------------
username = "root"
password = "2129"   # <-- put your password here
host = "localhost"
port = 3306


In [7]:
server_engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}"
)

with server_engine.connect() as connection:
    connection.execute(
        text("CREATE DATABASE IF NOT EXISTS hotel_analytics")
    )

print("Database created/exists.")

Database created/exists.


In [8]:
db_engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/hotel_analytics"
)

print("Connected to hotel_analytics!")

Connected to hotel_analytics!


In [9]:
hotel.to_sql(
    "hotel_bookings",
    con=db_engine,
    if_exists="replace",
    index=False,
    chunksize=5000
)

print("Data loaded successfully!")

Data loaded successfully!


In [10]:
pd.read_sql(
    "SELECT COUNT(*) AS total_rows FROM hotel_bookings",
    db_engine
)

,total_rows
0,87370


In [11]:
pd.read_sql(
    "SELECT * FROM hotel_bookings LIMIT 5",
    db_engine
)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,total_nights,total_guests,total_extra_requests
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01,0,2.0,0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01,0,2.0,0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-01,1,1.0,0
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-01,1,1.0,0
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,Transient,98.0,0,1,Check-Out,2015-07-03,2015-07-01,2,2.0,1
